# Teste de integração com MLflow

Experimento: `Telco_Churn` — run: `CatBoost_Vanilla`.

In [47]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [48]:
import pandas as pd


from projeto_ml.metricas import AvaliacaoModelo
from projeto_ml.pipeline import Pipeline_Data

from catboost import CatBoostClassifier, Pool
import mlflow
import mlflow.catboost

import warnings 

warnings.filterwarnings('ignore')


In [49]:
ORIGEM_DADOS = '../data/gold'
ALVO = ['Churn']
COLUNAS_EXCLUIR = ['customerID']

SEED  = 42


In [50]:
X_train = pd.read_parquet(f'{ORIGEM_DADOS}/train.parquet')
y_train= X_train['Churn']
X_train = X_train.drop(columns=['customerID', 'Churn'])


X_val = pd.read_parquet(f'{ORIGEM_DADOS}/val.parquet')
y_val = X_val['Churn']
X_val = X_val.drop(columns=['customerID', 'Churn'])


X_test = pd.read_parquet(f'{ORIGEM_DADOS}/test.parquet')
y_test= X_test['Churn']
X_test = X_test.drop(columns=['customerID', 'Churn'])


In [51]:
# 1. Configurando o experimento (só uma vez por projeto)
mlflow.set_experiment("Telco_Churn")

# 2. Abrindo o "envelope" da run
with mlflow.start_run(run_name="CatBoost_Feature_Engineering"):

    # --- FASE 2: TREINO ---
    CATEGORICAS = X_train.select_dtypes(include=['object', 'string']).columns.to_list()
    pool_train = Pool(X_train, y_train, cat_features=CATEGORICAS)
    pool_val = Pool(X_val, y_val, cat_features=CATEGORICAS)

    model = CatBoostClassifier(
        random_seed=42, 
        verbose=0, 


        depth = 3,
        iterations = 1500,
        early_stopping_rounds = 150,
        subsample = 0.80,
        bootstrap_type='MVS',
        rsm=0.80,
        max_ctr_complexity=2,
        use_best_model = True

        )
    model.fit(pool_train, eval_set=pool_val)

    # --- FASE 3: AVALIAÇÃO ---
    avaliacao = AvaliacaoModelo(
        modelo=model,
        X_train=X_train, y_train=y_train,
        X_holdout=X_test, y_holdout=y_test,
        nome_modelo="CatBoost Vanilla"
    )

    # --- FASE 4: O MLFLOW REGISTRA TUDO ---
    # 4.1. Logando Parâmetros
    mlflow.log_param("modelo", "CatBoost")
    mlflow.log_param("early_stopping", 50)
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param('depth', 3)
    mlflow.log_param('iterations', 1500)
    mlflow.log_param('early_stopping_rounds', 150)
    mlflow.log_param('subsample',.80)
    mlflow.log_param('boostrap_type', 'MVS')
    mlflow.log_param('rsm', .80)
    mlflow.log_param('max_ctr_complexity', 2)
    mlflow.log_param('use_best_model', True)

    # 4.2. Logando as Métricas (Extraídas da sua classe)
    df_metricas = avaliacao.calcular_metricas()

    mlflow.log_metric("roc_auc_holdout", df_metricas.loc[0, 'Holdout (Teste)'])
    mlflow.log_metric("f1_holdout", df_metricas.loc[3, 'Holdout (Teste)'])
    mlflow.log_metric("logloss_holdout", df_metricas.loc[2, 'Holdout (Teste)'])

    # 4.4. Logando o Modelo Treinado
    mlflow.catboost.log_model(model, name="modelo_catboost")

    # 5.0 REGISTRANDO OS GRÁFICOS DE PERFORMANCE 
    fig_roc = avaliacao.plotar_curva_roc()
    mlflow.log_figure(fig_roc, "curva_roc.png")

    fig_lift = avaliacao.plotar_curva_lift()
    mlflow.log_figure(fig_lift, "curva_lift.png")

    fig_matriz = avaliacao.plotar_matriz_confusao()
    mlflow.log_figure(fig_matriz, "matriz_confusao.png")

    fig_matriz = avaliacao.plotar_feature_importance()
    mlflow.log_figure(fig_matriz, "feature_importance.png")



Métricas de Avaliação (Holdout Final): CatBoost Vanilla


,Métrica,Treino,Holdout (Teste),Gap (Treino - Holdout)
0,ROC-AUC,0.8599,0.8427,0.0172
1,PR-AUC,0.6959,0.6681,0.0279
2,Logloss,0.3987,0.4175,-0.0188
3,F1-Score (Churn),0.6067,0.5504,0.0563
4,Precision (Churn),0.6901,0.6718,0.0183
5,Recall (Churn),0.5413,0.4662,0.0751
6,Acurácia,0.8138,0.7975,0.0163


2026/11/24 20:56:30 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
